# LLM Fine-tuning Notebook

LLM Fine-tuning Notebook for Llama-3.2-3B-Instruct with Unsloth

This notebook guides through the process of fine-tuning the Llama-3.2-3B-Instruct model
using the Unsloth library on Google Colab (T4 GPU).


# 1. Setup
Install necessary libraries and authenticate with Hugging Face.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install Unsloth and other dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unsloth/unsloth.git"
!pip install peft accelerate bitsandbytes transformers trl
!pip install huggingface_hub pyyaml


In [ ]:
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments, TextStreamer, AutoTokenizer
from trl import SFTTrainer
from peft import LoraConfig
import yaml
import os
from huggingface_hub import notebook_login, HfApi


In [ ]:
# Load parameters from params.yaml
with open("params.yaml", "r") as f:
    params = yaml.safe_load(f)

hf_model_repo = params["hf_model_repo"]
hf_lora_repo = params["hf_lora_repo"]
lora_r = params["lora_r"]
lora_alpha = params["lora_alpha"]
lora_dropout = params["lora_dropout"]
max_seq_length = params["max_seq_length"]
per_device_train_batch_size = params["per_device_train_batch_size"]
gradient_accumulation_steps = params["gradient_accumulation_steps"]
warmup_steps = params["warmup_steps"]
max_steps = params["max_steps"]
learning_rate = params["learning_rate"]
fp16 = params["fp16"]
logging_steps = params["logging_steps"]
output_dir = params["output_dir"]
optim = params["optim"]
seed = params["seed"]


In [ ]:
# Authenticate with Hugging Face
# You will be prompted to enter your Hugging Face token.
notebook_login()


# 2. Data Preparation
Load and preprocess the training dataset.


In [ ]:
from datasets import load_dataset

# Load your dataset
# Ensure your dataset.jsonl is in the `data/` directory
dataset = load_dataset("json", data_files={"train": "../data/dataset.jsonl"}, split="train")

# Define prompt template for fine-tuning
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Define an EOS token for the model
EOS_TOKEN = tokenizer.eos_token if tokenizer else "<|end_of_text|>"

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN
        text = alpaca_prompt.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched = True,)


# 3. Model Loading
Load the base language model and prepare it for LoRA fine-tuning.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = hf_model_repo,
    max_seq_length = max_seq_length,
    dtype = None, # None for auto detection. Use torch.bfloat16 for A100.
    load_in_4bit = True,
)

# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    bias = "none",
    use_gradient_checkpointing = "current_device",
    random_state = seed,
    max_seq_length = max_seq_length,
)


# 4. Tokenizer (Already handled in Model Loading, but shown as a separate step for clarity)
The tokenizer is loaded along with the model.


# 5. Training
Fine-tune the model using the prepared dataset.


In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can be True for faster training, but might lead to truncation issues
    args = TrainingArguments(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_steps = warmup_steps,
        max_steps = max_steps,
        learning_rate = learning_rate,
        fp16 = fp16,
        logging_steps = logging_steps,
        output_dir = output_dir,
        optim = optim,
        seed = seed,
    ),
)


In [ ]:
trainer.train()


# 6. Save LoRA adapter
Save the trained LoRA adapter locally.


In [ ]:
# Save the LoRA adapter locally
model.save_pretrained_merged("lora_adapter_merged", tokenizer)


# 7. Upload to Hugging Face Hub
Upload the trained LoRA adapter to Hugging Face Hub.


In [ ]:
# Push the LoRA adapter to Hugging Face Hub
# The model will be pushed to the repository specified in hf_lora_repo in params.yaml
model.push_to_hub(hf_lora_repo, token=True) # Set token=True if you authenticated via notebook_login()
tokenizer.push_to_hub(hf_lora_repo, token=True)

# Also push the merged model to Hub for easier use in inference
# model.save_pretrained_merged(hf_lora_repo, tokenizer, push_to_hub=True)


# 8. Inference (Example - for testing in Colab)
Quick test of the fine-tuned model.


In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "user", "content": "私は犬が好きです。これを否定文に変換してください。"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
